In [ ]:
# Purpose - assess RH conditions at different y cuts (with correct normalizations)

findex = 148 #143 #148 #143 #113 # 64  63 #120


In [ ]:
%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq
boilerplate = run.boilerplate + f" tstep = {run.steps[0]}"
cc = run.cc
B0 = run.B0

#my_steps = run.steps[:15]
#my_steps = [run.steps[-1]]
my_steps = run.steps
#my_steps = run.steps[::10]
#my_steps = [99]
nsteps = len(my_steps)

print(run.steps)

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = True)

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim
rect = [0, run.xlim, 0, run.ylim]

xstrips = [None] * nsteps

xcut_y = 1000
xcut_y = 800
xcut_y = 1600

for ts in range(nsteps):

    roi = fq_cut(d.fields, ts, rect)

    #xcut_y = ylim / 4 * 0.75
    ycut_x = xlim / 2

    xstrips[ts] = fq_xcut(roi, xcut_y)

In [ ]:
if False:
    tchosen = 17
    #roi = fq_cut(d.fields, 0, [80, 160, 0, 200])
    roi = fq_cut(d.fields, tchosen, [0, run.xlim, 0, run.ylim])

    #roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

    # Create a subplot with one plot
    fig, ax = plt.subplots()

    # Generate some sample data for the scatter plot
    x = np.random.rand(100)
    y = np.random.rand(100)

    plt.title(f"{run.boilerplate}")
    plot_basic(ax, run, roi, sdens = 0.5, cutoff = 5)
    plt.axhline(y = xcut_y, color = 'g', linestyle = '--')

    # Show the plot
    plt.show()

In [ ]:
# Shock Conditions

angle_in_deg = 4.565 #4

theta = angle_in_deg * 0.5 * np.pi / 180

nx, ny, nz = -np.cos(theta), -np.sin(theta), 0
tx, ty, tz = np.sin(theta), -np.cos(theta), 0 

plt.figure()
plt.quiver(0, 0, nx, ny, color='r', scale=1, scale_units='xy', angles='xy', label='n')
plt.quiver(0, 0, tx, ty, color='b', scale=1, scale_units='xy', angles='xy', label='t')
plt.xlim(-1, 1); plt.ylim(-1, 1)
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.show()

PTI = 5.0 / 3.0 # polytropic index

# n cross t is positive

In [ ]:
strip = sum(xstrips) / len(xstrips)

In [ ]:
plt.figure()
plt.plot(strip.x, strip.pb)
plt.title("Magnetic Pressure (pb)")
plt.xlabel("x")
plt.ylabel("pb")
plt.show()

crossing_index = np.where(np.diff(strip.pb > 0.5))[0][1]
crossing_x = strip.x[crossing_index]
print(f"pb first crosses 0.5 at x = {crossing_x.values}")

In [ ]:
# RH Conditions

lim1 = 400.0
lim2 = 460.0

BUFFER = 15

#lim1 = run.xlim / 2
#lim2 = 0.95 * run.xlim

def plot_squant(axis, quantity):
    global lim2
    right_val = strip[quantity].sel(x = lim2, method="nearest")
    axis.plot(strip.x, strip[quantity] / right_val)

    #try:
        #right_val = strip[quantity + '_r'].sel(x = lim2, method="nearest")
    #    axis.plot(strip.x, strip[quantity + '_r'] / right_val, color = 'red')
    #except KeyError:
    #    pass

strip['v_n'] = strip.velx * nx + strip.vely * ny + strip.velz * nz # in units of c
strip['v_t'] = strip.velx * tx + strip.vely * ty + strip.velz * tz
strip['B_n'] = (strip.bx * nx + strip.by * ny + strip.bz * nz)
strip['B_t'] = (strip.bx * tx + strip.by * ty + strip.bz * tz)  
strip['vtot'] = np.sqrt(strip.velx ** 2 + strip.vely ** 2 + strip.velz ** 2)
strip['densm'] = strip.dens + 0.5 * run.sigma * strip.pb
#strip['densm'] = 0 * strip.dens

strip['pn'] = strip.px * nx ** 2 + strip.py * ny ** 2

strip['v_dot_B'] = strip.bx * strip.velx + strip.by * strip.vely + strip.bz * strip.velz
strip['v_sqrd'] = strip.velx ** 2 + strip.vely ** 2 + strip.velz ** 2

PNORM = 2 / run.sigma

fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax_dens, ax_rh1, ax_rh2, ax_rh3, ax_vn, ax_rh4 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

plt.subplots_adjust(hspace=0, wspace = 0.2)


eqn_fsize = 18

# Mass

strip['mass_flux'] = strip.dens * strip.v_n 
strip['mass_flux_r'] = strip.densm * strip.v_n 

plot_squant(ax_dens, 'mass_flux')
#ax1.set_title(r"$\rho v_n$")
ax_dens.text(0.45, 0.10, r"$\rho v_n$", transform=ax_dens.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Pressure

ram = PNORM * strip.densm * strip.v_n ** 2
strip['pressure'] = ram + strip.pn + strip.pb #changed from px


# TODO fix px (think this has been fixed)

plot_squant(ax_rh1, 'pressure')
#ax2.set_title(r"$\rho v_n^2 + P_g + P_B$")
ax_rh1.set_ylim(0, 1.2)
ax_rh1.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax_rh1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')


# Energy

strip['energy'] = strip.v_n * (0.5 * PNORM * strip.dens * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

strip['energy_r'] = strip.v_n * (0.5 * PNORM * strip.densm * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

plot_squant(ax_rh2, 'energy')
#ax3.set_title("Energy")

energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax_rh2.text(0.15, 0.10, energy_expression, transform=ax_rh2.transAxes, fontsize=12, verticalalignment='bottom')

# B_N

plot_squant(ax_rh3, 'B_n')
#ax4.set_title("$B_n$")
ax_rh3.text(0.45, 0.10, '$B_n$', transform=ax_rh3.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Poynting Flux

strip['poynting'] = PNORM * strip.dens * strip.v_n * strip.v_t - 2 * strip.B_n * strip.B_t
strip['poynting_r'] = PNORM * strip.densm * strip.v_n * strip.v_t / np.sqrt(1 - strip.vtot ** 2) - 2 * strip.B_n * strip.B_t
plot_squant(ax_vn, 'poynting')
#ax5.set_title(r"$\rho v_n v_t - B_n B_t / 4 \pi$")
ax_vn.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax_vn.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Electric Field

strip['E_ideal'] = -strip.v_n * strip.B_t + strip.v_t * strip.B_n # I flipped this so it is Ez_ideal

plot_squant(ax_rh4, 'E_ideal')
ax_rh4.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax_rh4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

#ax6.set_title("$B_n v_t - v_n B_t$")



for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(lim1, lim2)
        ax.set_ylim(0, 2.4)

        #ax.axvspan(strip.x[ends[0]], strip.x[ends[1]], color='gray', alpha=0.1, zorder=0)

for ax in [ax_rh2, ax_rh4]:
    ax.set_xlabel("$x$ [$c / \omega_p$]")

#plt.suptitle("Rankine-Hugoniot Conditions \n" + run.boilerplate + f"\n steps = {my_steps}, y = {xcut_y} ({xcut_y / run.ylim:.02f})")
#plt.suptitle("Rankine-Hugoniot Conditions")


plt.show()

#save_polished_fig("rh")

print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])

In [ ]:
# PLOT NUMBER 2

background_color = '#f5f5f5'


BUFFER = 15

ends = []

for val in [0.15, 0.85]:
    boundary = np.where(np.diff(strip.pb > val))[0][1]
    print(boundary)
    ends.append(boundary)



lim1 = ends[0] - BUFFER
lim2 = ends[1] + BUFFER

def plot_squant(axis, quantity, is_rh = True, label = None):
    global lim2
    if is_rh:
        #return
        axis.axhline(1, color='gray', linestyle=':', linewidth=1)
        right_val = strip[quantity].sel(x = lim2, method="nearest")
        axis.plot(strip.x, strip[quantity] / right_val)
    else:
        axis.plot(strip.x, strip[quantity], color = '#d62728', label = label)

fig, axes = plt.subplots(4, 2, figsize=(8, 5.5), sharex=True)
ax_dens, ax_rh1, ax_rh3, ax_rh5, ax_vn, ax_rh2, ax_rh4, ax_rh6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[3, 0], axes[0, 1], axes[1, 1], axes[2, 1], axes[3, 1]

plt.subplots_adjust(hspace=0, wspace=0.2)

eqn_fsize = 18

strip['v_n_norm'] = strip.v_n / (0.1 * run.beta_A)

# Move density and v_n to the top row
plot_squant(ax_dens, 'dens', is_rh = False, label = '$n / n_0$')
ax_dens.text(0.45, 0.10, r'$n / n_0$', transform=ax_dens.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
#ax1.legend()

plot_squant(ax_vn, 'v_n_norm', is_rh = False, label = '$v_n / (0.1 v_A)$')
ax_vn.text(0.45, 0.10, r'$v_n / (0.1 v_A)$', transform=ax_vn.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
#ax5.legend()

# Mass
strip['mass_flux'] = strip.dens * strip.v_n 
strip['mass_flux_r'] = strip.densm * strip.v_n 
plot_squant(ax_rh1, 'mass_flux')
ax_rh1.text(0.45, 0.10, r"$\rho v_n$", transform=ax_rh1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Pressure
ram = PNORM * strip.densm * strip.v_n ** 2
strip['pressure'] = ram + strip.pn + strip.pb
plot_squant(ax_rh2, 'pressure')
ax_rh2.set_ylim(0, 1.2)
ax_rh2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax_rh2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Energy
strip['energy'] = strip.v_n * (0.5 * PNORM * strip.dens * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

strip['energy_r'] = strip.v_n * (0.5 * PNORM * strip.densm * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

plot_squant(ax_rh3, 'energy')
energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax_rh3.text(0.15, 0.10, energy_expression, transform=ax_rh3.transAxes, fontsize=12, verticalalignment='bottom')

# B_N
plot_squant(ax_rh4, 'B_n')
ax_rh4.text(0.45, 0.10, '$B_n$', transform=ax_rh4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Poynting Flux
strip['poynting'] = PNORM * strip.dens * strip.v_n * strip.v_t - 2 * strip.B_n * strip.B_t
strip['poynting_r'] = PNORM * strip.densm * strip.v_n * strip.v_t / np.sqrt(1 - strip.vtot ** 2) - 2 * strip.B_n * strip.B_t
plot_squant(ax_rh5, 'poynting')
ax_rh5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax_rh5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Electric Field
strip['E_ideal'] = -strip.v_n * strip.B_t + strip.v_t * strip.B_n
plot_squant(ax_rh6, 'E_ideal')
ax_rh6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax_rh6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlim(lim1, lim2)
        ax.set_ylim(0, 2.4)

        #ax.axvspan(strip.x[ends[0]], strip.x[ends[1]], color='blue', alpha=0.1, zorder=0)
        #ax.set_facecolor(background_color)

ax_dens.set_ylim(0, 3)
ax_vn.set_ylim(0, 1)

for ax in [ax_rh3, ax_rh6, ax_dens, ax_vn]:
    ax.set_xlabel("$x$ [$c / \omega_p$]")

plt.show()

print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])

save_polished_fig("rh")


In [ ]:
energy_expression = r'\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}'
energy_expression = r'$\rho v$'

plt.figure()
plt.title(energy_expression)
plt.show()

In [ ]:
# testing

plt.rcParams.update({'font.size': 18})

max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max()

naive_mass_flux = strip.dens * strip.v_n / (run.beta_A * 0.1)
#relativistic_mass_flux = (strip.dens + 0.5 * run.sigma * (strip.B_tot ** 2 + PTI / (PTI - 1) * strip.p))* strip.v_n / (run.beta_A * 0.1)
good_enough = strip.densm * strip.v_n / (run.beta_A * 0.1)


plt.figure()
#plt.axvline(boundary, color = 'green', linestyle = '--')

#for lim in [lim1, lim2]:
    #plt.axvline(lim, color = 'green', linestyle = '--')

plt.axvspan(lim1, lim2, color='grey', alpha=0.2)

#plt.plot(strip.x, strip.dens, label = "$n / n_0$")
#plt.plot(strip.x, strip.v_n / (run.beta_A * 0.1), label = "$v_x / v_{A0}$")
#plt.plot(strip.x, relativistic_mass_flux, label = "with B inertia")
plt.plot(strip.x, naive_mass_flux, label = "without B inertia")
plt.plot(strip.x, good_enough, label = "good enough")

#plt.plot(strip.x, strip.epsilon - 1, label = "$\epsilon$")
plt.plot()

plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

In [ ]:
plt.rcParams.update({'font.size': 18})

max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max()

boundary = np.where(np.diff(strip.pb > 0.5))[0][1]

plt.figure()

#for end in ends:
#    plt.axvline(end, color = 'green', linestyle = '--')

plt.axvspan(lim1, lim2, color='grey', alpha=0.2)


plt.plot(strip.x, strip.dens, label = "$n / n_0$")
plt.plot(strip.x, strip.pn, label = "$P_g / P_{B0}$")
plt.plot(strip.x, strip.pb, label = "$P_B / P_{B0}$")
plt.plot(strip.x, strip.jz / max_jz, label = "$j_z / j_{\mathrm{max}}$")
#plt.gca().set_facecolor(background_color)
#plt.plot(strip.x, strip.v_n / (0.1 * 0.5), label = "$v_x / v_{A0}$")
#plt.plot(strip.x, strip.v_n * strip.dens, label = "$n v_x / v_{A0}$")

plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")

plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
plt.rcParams.update({'font.size': 18})

max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max()

boundary = np.where(np.diff(strip.pb > 0.5))[0][1]

plt.figure()

#for end in ends:
#    plt.axvline(end, color = 'green', linestyle = '--')

plt.axvspan(lim1, lim2, color='grey', alpha=0.2)

strip["ez_ni"] = strip.ez + (strip.velx * strip.by - strip.vely * strip.bx)

plt.plot(strip.x, strip.dens, label = "$n / n_0$")
plt.plot(strip.x, strip.pn, label = "$P_g / P_{B0}$")
plt.plot(strip.x, strip.pb, label = "$P_B / P_{B0}$")
plt.plot(strip.x, strip.jz / max_jz, label = "$j_z / j_{\mathrm{max}}$")
plt.plot(strip.x, 200 * strip.ez_ni, label = "$200 E_{ni}$ ")
#plt.plot(strip.x, strip.ez_ni / strip.jz, label = "$\eta$")
#plt.gca().set_facecolor(background_color)
#plt.plot(strip.x, strip.v_n / (0.1 * 0.5), label = "$v_x / v_{A0}$")
#plt.plot(strip.x, strip.v_n * strip.dens, label = "$n v_x / v_{A0}$")

ez_sum_da = strip.ez_ni.sel(x=slice(lim1, lim2)).sum(dim='x')
jz_sum_da = strip.jz.sel(x=slice(lim1, lim2)).sum(dim='x')

try:
    ez_sum = float(ez_sum_da.compute().item())
    jz_sum = float(jz_sum_da.compute().item())
except Exception:
    ez_sum = float(ez_sum_da.item())
    jz_sum = float(jz_sum_da.item())

if jz_sum == 0:
    print("sum(jz) is zero, quotient undefined")
else:
    print(f"sum(ez_ni) = {ez_sum}, sum(jz) = {jz_sum}, quotient = {ez_sum / jz_sum}")

plt.ylim(-0.5, 4)

plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")

plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
# Slow-Mode Shock Conditions

fig, axes = plt.subplots(3, 2, figsize=(8, 6))
ax_dens, ax_rh1, ax_rh2, ax_rh3, ax_vn, ax_rh4 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

def degree_formatter(x, pos):
    return f"{x:.0f}°"

strip['B_mag'] = strip.bx ** 2 + strip.by ** 2 + strip.bz ** 2
strip['theta_B'] = np.arccos(strip.B_n / strip.B_mag)

strip['v_A'] = np.sqrt(run.sigma) * strip.B_mag
strip['v_I'] = strip.v_A * np.cos(strip.theta_B)
strip['v_S'] = np.sqrt(run.sigma / 2 * PTI * strip.px / strip.dens)
strip['v_SM'] = np.sqrt(0.5) * np.sqrt(strip.v_S ** 2 + strip.v_A ** 2 - \
                                       np.sqrt((strip.v_S ** 2 + strip.v_A ** 2) ** 2 - 4 * strip.v_S ** 2 * strip.v_I ** 2))

strip['M_I'] = strip.v_n / strip.v_I 
strip['M_SM'] = strip.v_n / strip.v_SM

# Plasma Pressure

plot_squant(ax_dens, 'px')
ax_dens.set_title(r"$P_x$ (increase)")

# Magnetic Pressure

plot_squant(ax_rh1, 'pb')
ax_rh1.set_title(r"$P_B$ (decrease)")
ax_rh1.set_ylim(0)

# Angle
strip['theta_B_deg'] = 180 / np.pi * np.arccos(strip.B_n / strip.B_mag)

plot_squant(ax_rh2, 'theta_B_deg')
ax_rh2.set_title(r"$\theta_B$ (decrease)")
ax_rh2.yaxis.set_major_formatter(FuncFormatter(degree_formatter))

# M_I

plot_squant(ax_rh3, 'M_I')
ax_rh3.set_title(r"$M_I$ ($\leq 1$ upstream)")

# M_SM

plot_squant(ax_vn, 'M_SM')
ax_vn.set_title(r"$M_{SM}$ ($> 1 \; \rightarrow \; < 1$)")

for ax in [ax_rh3, ax_vn]:
    ax.axhline(1, color = 'black', linestyle = '--')
    ax.axhline(0, color = 'grey', linestyle = ':', alpha = 1)

for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(run.xlim / 2, run.xlim)

        for bound in strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)]:
            ax.axvline(bound, color = 'green', linestyle = '--')



plt.suptitle("Slow-Mode Shock Conditions \n" + run.boilerplate + f"\n steps = {my_steps}")

plt.tight_layout()
plt.show()



print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])